# Plotting Recommendations for Chem 4410 - Part 3

We have seen how to plot linear and non-linear models. This notebook will set an example of a typical workflow for these kinds of data analysis. This will likely be the notebook that you will want to copy and reuse. It will not generate data, like the previous two notebooks. We will start with a file of experimental data, create a model and fit the data to the model. Then we will report the parameters and output three versions of the line fit plots.

## Step 1: Setup the Tools

The code below will load in the modules, packages and set global variables for all the code blocks that follow. Much of that code is specific for running this notebook in Colab. We need to install packages that are not included in Colab and load in a data file and a plotting style file to use later.

In [ ]:
# Install packages only if needed
# Coogle Colab does not include every Python package
#   
try:                  # if the package exists import it, otherwise install it
    import lmfit
except ImportError:
    !pip -q install lmfit
    import lmfit

try:
    import uncertainties
except ImportError:
    !pip -q install uncertainties
    import uncertainties

# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import uncertainties as un
from uncertainties import unumpy as unp

from matplotlib.ticker import FormatStrFormatter
from scipy.stats import t

from pathlib import Path

Path("plots").mkdir(exist_ok=True)
Path("data").mkdir(exist_ok=True)

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not Path("data/3c.csv").exists():
    !wget -q -O data/3c.csv https://raw.githubusercontent.com/blinkletter/Chem4410Webbook/main/book/notebooks/M11_Plot_Recommendations/3c.csv

if IN_COLAB and not Path("tufte.mplstyle").exists():
    !wget -q https://raw.githubusercontent.com/blinkletter/Chem4410Webbook/main/book/styles/tufte.mplstyle


plt.rcdefaults()

print("Setup complete")


### Load the Data

The code below will import the text from the csv file and create a new dataframe containing the data. Inspect the data and identify the columns that we will use as x and y. 

Note: I am using data from Wang, F; Finnin, J; et al., "The Hydrolysis of Diclofenac Esters: Synthetic Prodrug Building Blocks for Biodegradable Drug–Polymer Conjugates." *J. Pharm. Sci.*, **2016**, *105*, 773-785.  [https://doi.org/10.1002/jps.24665](https://doi.org/10.1002/jps.24665). this data is presented, analyzed and discussed in the "Carbonate Ester Hydrolysis" that was presented in today's class meeting. 


In [ ]:
filename = "data/3c.csv"  # the data set oif a pH-rate profile

# read in the file and convert it to a DataFrame using pandas read_csv tool
df = pd.read_csv(filename,
                 index_col=False,          # all columns will be regular columns (for consistancy)
                 skipinitialspace = True,  # ignore spaces before each data entry
                 comment = "#")            # ignore comments (lines that start with "#")

print(df.head())                           # print just the first few lines

### Calculations

I can use the data directly and convert the units of the result or I can convert the units now and then use the data to produce a result with the desired units. I see that the rate constants reported at each pH value are in units of $\text{hours}^{-1}$. I will convert it to $\text{seconds}^{-1}$. We will be plotting $\log{k_{obs}}$ vs $pH$, and so will will also need to apply the $\log _{10}$ function.

You will have to set up calculations unique to your data set and experiment. the code below can be used as a starting point.

The code below will set our $x$ and $y$ values.

In [ ]:
x = df["pH"]                  # use the column headings to extract each column as an array
k_obs = df["k1 (10-5 hr-1)"]

k_obs_seconds = k_obs /3600   # There are 3600 seconds in each hour
y = np.log10(k_obs_seconds)   # y is log10(k_obs)

### Inspect the Data

Always make a quick inspection of you data before spending any time on analysis. Does it appear to be quality data? Does it appear to match your hypothesis? 

In [ ]:
# Plot the raw data
plt.figure(figsize=(4, 3.5))                  # set the size for smaller plots

# Plot
plt.scatter(                                  # the scatter function will make a classic scatter plot
            x, y,                             # a set of x,y data to plot
            color="navy",                     # many options are available. here I set the color
            s=50,                             # size of the data points. 
            label="Experimental data",        # name of the data set (will be used in a legend)
            )                                 # there are many more option available

#plt.axhline(0)                                # Add a line at y=0 (to make a point about data analysis later)

plt.show()                                    # show the plot

## Using *LMFit*

We are now ready to curve fit the data. 

The model for this data is the sum of three different rates. There is an acid-dependant rate (observed rate gets faster as acid increases), a base-dependant rate (observed rate gets faster as base increases), and a constant background rate (often called the "water rate").

\begin{align*}
k_{obs_{H^+}} &= k_{H^+}\left[ H^+ \right] \\
k_{obs_{H_2O}} &= k_{H_2O} \\
k_{obs_{OH^-}} &= k_{OH^-}\left[ OH^- \right] \\
\end{align*}

and

$$\left[ OH^- \right] = \frac{K_W}{\left[ H^+ \right]}$$

We can create a model by adding the three rates together to get the observed rate constant at any given $pH$ value.

\begin{align*}
k_{obs} &= k_{obs_{H^+}} + k_{obs_{H_2O}} + k_{obs_{OH^-}} \\
        &= k_{H^+}\left[ H^+ \right] + k_{H_2O} + k_{OH^-}\frac{K_W}{\left[ H^+ \right]}
\end{align*}

This model gives us $K_{obs}$ in terms of \left[ H^+ \right]$$. The model requires the parameters $k_{H^+}$, k_{H_2O}, and k_{OH^-} to be optimized until the fit best matches the data. The parameter $K_W$ is a fixed value of $10^{-14}$.

We will set up a model function that uses all four parameters. We will then create a parameter set where one of the parameters, $K_W$, is fixed at a value of $10^{-14}$ and allow the others to vary in the optimization.

### A Note on Copy \& Paste

Almots all of the code in this notebook was copied and pasted from the previous notebook. I changed the model and changed some text in the `lmfit.fit()` function call. take note of where it uses `pH = x`. The variable pH is used in the function and we assugn the values of x to it.


In [ ]:
# define the function for the model
def pH_rate(pH, k_H, k_H2O, k_OH):
    K_W = 10**-14
    H = 10**(-pH)
    OH = K_W / H
    k_obs = k_H * H + k_H2O + k_OH * OH
    log_k_obs = np.log10(k_obs)
    return log_k_obs

# Create a model by loading the function via the lmfit.Model tool
model = lmfit.Model(pH_rate)  

# Set parameters - I can make a variable into a constant by setting vary=False.
params = model.make_params(
    k_H = dict(
        value = 1E1,      # setting an initial guess. This is not necessary but can help in complex cases.
        vary = True,    # allow the parameter to vary in the optimization
    ),
    k_H2O = dict(       # 
        value = 1E-2,      # set A_inf to zero
        vary = True,    # allow the parameter to vary in the optimization
    ), 
    k_OH =  dict(
        value = 1E7, 
        vary = True,    # allow the parameter to vary in the optimization
   ),
)

# use the .fit method on the model object to perform the curve fit
result = model.fit(y, params, pH=x) 

print(result.fit_report())
print()
#print(result.ci_report())



### A Quick Plot

The entire curve fit was in the code block above and the visual output of the curve fit is in the code block below. You can see that there is not much typing involved.


In [ ]:
plt.rcdefaults()
plt.rcParams["figure.figsize"] = (4, 3.5)    # set some default style paramers for plots
result.plot(numpoints=100)                   # higher values for numpoints results in smoother curve fit

plt.show()

### Calculate Line Fit and Intervals

The code below will make a set of x-data with many points so that we can calculate a smooth curve fit. We will calculate y-values for eact x value using the parameters from the curve fit. *LMFit* can do this from the `result` object using the built-in `result.eval()` function.

*LMFit* can also calculate the standard error at every x-value. Below I use the `result.eval_uncertainty()` function to calculate the $2\sigma$ strandard error (approx. 95% confidence). Adding and subtracting this value from the line fit will give us a range where we would expect the curve fit to be 95% of the time over millions of similar experiments.

The prediction interval is the range in which we would expect experimental data points to appear over many, many repeated experiments. It is obtained by estimating the standard error of the population of the data (a score for how scattered the data points are) and combining that with the errors determined for the fit parameters. It is sometimes useful to present this range on the residual plot to help identify possible outliers.

In [ ]:

# line fit
x_fit = np.linspace(np.min(x)-0.5, np.max(x)+0.5, 50)     # make an array of 50 points that span the data range
y_fit = result.eval(pH=x_fit)                      # the line of the line fit

# confidence interval
y_err = result.eval_uncertainty(pH=x_fit, sigma=2) # the 95% confidence interval
ci_upper = y_fit + y_err
ci_lower = y_fit - y_err

# Prediction interval

se = result.eval_uncertainty(pH=x_fit, sigma=1)  # standard error of the fit
mse = result.redchi                             # reduced chi-squared value from the fit (mean squared error)
se_pred = np.sqrt(se**2 + mse)

dof = result.ndata - result.nvarys # degrees of freedom = number of data points - number of fitted parameters
tval = t.ppf(0.975,dof)            # two-tailed t-value for 95% confidence interval
pi = tval * se_pred

pi_lower = y_fit - pi
pi_upper = y_fit + pi


## Visualizing the Line Fit

In the code blocks below I will present code to plot the line fit, with its 95% confidence interval; the residual plot with the confidence interval and the prediction interval shown; and a combined plot that includes both the line fit and the residual plot in a single figure. Each plot is output as a pdf file for documents and a png file for web pages.

### Line Fit

The code below will output a plot of the data points, the best-fit line and the $2\sigma$ (~95%) confidence interval for the line fit. 

In [ ]:
# Fancy Plotting
# Note: We have x, y, x_fit, y_fit, ci_upper and ci_lower from previous calculations in the code blocks above.

from matplotlib.ticker import FormatStrFormatter

plt.rcdefaults()                 # reset plotting style
plt.style.use("tufte.mplstyle")  # apply the style sheet

fig, ax = plt.subplots(figsize = (4, 3.5))

ax.scatter(
    x, y, 
    marker = "o", s=32, 
    c="white", edgecolor = "black",
    linewidth = 0.7, 
    label=r"$\log{k_{obs}}$", 
    zorder=3
    )

ax.plot(
    x_fit, y_fit,
    color="black", 
    linewidth=0.5, zorder=2,
    )

ax.fill_between(
    x_fit, ci_lower, ci_upper,
    linewidth = 0,
    facecolor="gray", 
    edgecolor = "gray", 
    alpha=0.20,
    label="95% Confidence Band",
    zorder = 2,
    )

ax.spines["left"].set_position(("outward", 8))
ax.spines["bottom"].set_position(("outward", 8))
ax.set(
    ylabel=r"$\log{(k_{obs}\;/s^{-1})}$", 
    xlabel=r"$\text{pH}$",
    xlim=[0,9], 
    xticks = [0,2,4,6,8],                 
#    ylim=[-.2,3],     
    yticks = [-2,-1,0,1,2],                 
      )
ax.yaxis.set_major_formatter(FormatStrFormatter('%.1f')) # 1 decimal place

fig.tight_layout()
fig.savefig("plots/result2.pdf")
ax.patch.set_facecolor([0, 0, 0, 0])          # Set face of plot to transparent
fig.savefig(f"plots/result2.png", dpi=600,    # save plot as .png with transparent background
            facecolor = [0, 0, 0, 0],
        )

plt.show()



### Residual Plot

The code below will output a plot of the residuals, the $2\sigma$ (~95%) confidence interval for the line fit and the 95% prediction interval for the data set. It is sized to fit the style of my dosuments. You can change the size however you like.

In [ ]:

# Residual Plot

x = result.userkws["pH"]      # original x data: we sent in a set labeles as "x"
y = result.data              # original y data
residuals = result.residual  # residuals 

################################################################################

fig, ax = plt.subplots(figsize = (3, 2.5))

ax.scatter(
    x, residuals, 
    marker = "o", s=32, 
    c="white", edgecolor = "black",
    linewidth = 0.7, 
    label=r"$\log{k_{obs}}$", 
    zorder=4
    )

ax.hlines(
    0, np.min(x_fit), np.max(x_fit),
    color="black", 
    linewidth=0.5, zorder=3,
    )

ax.fill_between(
    x_fit, -y_err, y_err,
    linewidth = 0,
    facecolor="gray", 
    edgecolor = "gray", 
    alpha=0.20,
    label="95% Confidence Band",
    zorder = 2,
    )

ax.fill_between(
    x_fit, -pi, pi,
    linewidth = 0,
    facecolor="gray", 
    edgecolor = "gray", 
    alpha=0.1,
    label="95% prediction Band",
    zorder = 1,
    )

ax.spines["left"].set_position(("outward", 8))
ax.spines["bottom"].set_position(("outward", 8))

res_span = np.max(np.abs(residuals)) * 2
ax.set(
       ylabel=r"$\Delta\log{(k_{obs}\;/s^{-1})}$", 
       xlabel=r"$\text{pH}$",
#       xlim=[0,43], 
       xticks = [1,3,5,7],                 
#       ylim=[-0.4, 0.4],     
#       ylim=[-res_span, res_span],     
#       yticks = [-0.3, 0, 0.3],                 
      )

fig.tight_layout()
fig.savefig("plots/result3.pdf")              # save plot as .pdf
ax.patch.set_facecolor([0, 0, 0, 0])          # Set face of plot to transparent
fig.savefig(f"plots/result3.png", dpi=600,    # save plot as .png with transparent background
            facecolor = [0, 0, 0, 0],
        )

plt.show()

### Combined Plot

The code below will output a plot of the resisuals, the $2\sigma$ (~95%) confidence interval for the line fit and the 95% prediction interval for the data set. It is sized to fit the style of my dosuments. You can change the size however you like.

In [ ]:
from matplotlib.ticker import FormatStrFormatter

plt.rcdefaults()
plt.style.use("tufte.mplstyle")

fig, ax = plt.subplots(nrows=2, ncols=1, figsize=(4,5), height_ratios=[1, 4])  

# plot data points
ax[1].scatter(
    x, y, 
    marker = "o", s=32, 
    c="white", edgecolor = "black",
    linewidth = 0.7, 
    label=r"$\ln{(\text{conc }/M)}$", 
    zorder=3
    )

# plot smooth line fit
ax[1].plot(
    x_fit, y_fit,
    color="black", 
    linewidth=0.5, zorder=2,
    )

# plot confidence interval for line fit
ax[1].fill_between(
    x_fit, ci_lower, ci_upper,
    linewidth = 0,
    facecolor="gray", 
    edgecolor = "gray", 
    alpha=0.20,
    label="95% Confidence Band",
    zorder = 2,
    )

# Settings for main plot
ax[1].set(
#   title = "Title",   
    ylabel=r"$\log{(k_{obs}\;/s^{-1})}$", 
    xlabel=r"$\text{pH}$", #   xlim=[43], 
    xticks = [0,2,4,6,8],                 
 #   ylim=[-2,3],     
 #   yticks = [-8.0,-7.0,-6.0],                 
       )

# add line at x=0 to compare how the value for A_inf sets the endpoint
if False:              # flag to turn this on or off
    ax[1].hlines(
        0, np.min(x_fit), np.max(x_fit),
        color="red", 
        linewidth=0.3, zorder=0,
        )

ax[1].yaxis.set_major_formatter(FormatStrFormatter('%.1f')) # 1 decimal place in y axis

ax[1].spines["left"].set_position(("outward", 8))
ax[1].spines["bottom"].set_position(("outward", 8))

######################
### Plot the residuals
######################

# plot residuas
ax[0].scatter(
    x, residuals, 
    marker = "o", s=32, 
    c="white", edgecolor = "black",
    linewidth = 0.7, 
    label=r"residuals", 
    zorder=4
    )

# make line along zero for residual
ax[0].hlines(
    0, np.min(x_fit), np.max(x_fit),
    color="black", 
    linewidth=0.5, zorder=3,
    )

# add residual confidence interval
ax[0].fill_between(
    x_fit, -y_err, y_err,
    linewidth = 0,
    facecolor="gray", 
    edgecolor = "gray", 
    alpha=0.20,
    label="95% Confidence Band",
    zorder = 2,
    )

# add residual prediction interval
ax[0].fill_between(
    x_fit, -pi, pi,
    linewidth = 0,
    facecolor="gray", 
    edgecolor = "gray", 
    alpha=0.1,
    label="95% prediction Band",
    zorder = 1,
    )

# settings for residual plot
res_span = np.max(np.abs(residuals)) * 2
ax[0].set(
    ylabel=r"$\text{residuals}$", 
    xlabel=r"",
 #  xlim=[0,43], 
 #   xticks = [0,5,10,15],                 
 #   ylim=[-res_span, res_span],
 #   ylim=[-0.4, 0.4],
    yticks = [-0.2, 0, 0.2],                 
    )

ax[0].spines["left"].set_position(("outward", 8))
#ax[0].spines["bottom"].set_position(("outward", 8))
ax[0].set_xticks([])


################################################
### Output Plot
################################################

# Plot as .pdf
fig.savefig(f"plots/result4.pdf")

### Set face of plot to transparent
ax[0].patch.set_facecolor([0, 0, 0, 0])  
ax[1].patch.set_facecolor([0, 0, 0, 0])  

# Plot as .png with transparent background
fig.savefig(f"plots/result4.png", dpi=600, 
            facecolor = [0, 0, 0, 0],
        )
# display plot in notebook
plt.show()

print(result.fit_report())
print()
print(result.ci_report())
